<div class="alert alert-block" style="border: 1px solid #1E90FF;background-color:#BECDDD;font-size:1.2em;line-height:1.4em">
- 강의자료를 무단으로 전재, 방송, 복사, 배포, 전시, 판매, 왜곡, 변조, 개작하는 것은 금지되어 있습니다.<br>
</div>

## 13주차 최단 경로 알고리즘

#### 1. 최단 경로 알고리즘
- 그래프(graph)에서 두 노드를 잇는 간선의 가중치 합이 최소가 되는 경로를 찾는 것

#### 다익스트라 알고리즘
- 다익스트라 알고리즘은 단일 출발 최단 경로 문제 종류임
  - 하나의 정점에서 다른 모든 정점에 도착하는 최단 경로를 구하는 문제

![nn](img/img_13_01.png)

#### 다익스트라 알고리즘 로직
- 첫 정점을 기준으로 연결되어 있는 정점들을 차례로 추가해서, 최단 거리를 업데이트하는 기법
- 다익스트라 알고리즘은 너비우선탐색(BFS)와 유사함
  - 첫 정점부터 각 노드간의 거리를 저장하는 배열을 만들고, 인접 노드 간의 거리를 계산하면서 최단 거리를 해당 배열에 업데이트를 진행함

- 우선순위 큐를 활용한 다익스트라 알고리즘
  - 우선순위 큐는 최소 힙 방식을 통해 가장 짧은 거리를 가진 노드 정보를 먼저 확인함

####  우선순위 큐와 정렬
- 우선순위 큐는 내부적으로 정렬 기능을 수행됨
- 따라서 PriorityQueue 에 입력되는 객체는 객체간의 정렬을 위한 기준을 정의해야 함
- 우선순위 큐 내부에서 정렬을 사용할 때는, 해당 객체의 Comparable 인터페이스의 compareTo() 메서드를 호출을 위해 이를 구현해야 함

In [228]:
// 10주차 그래프 edge 구현
public class Edge implements Comparable<Edge> {
    public int route;
    public String node;
    
    public Edge (int route, String node) {
        this.route = route;
        this.node = node;
    }
    
    public String toString() {
        return "node: " + this.node + ", route: " + this.route;
         
    }
    @Override
    public int compareTo(Edge edge) {
        return this.route - edge.route;
    }
}

#### 우선순위 큐
- java.util.PriorityQueue 사용함

> 우선순위큐는 데이터를 입력 후, 우선순위에 데이터가 출력 되면서, 해당 데이터는 큐에서 삭제됨

In [235]:
import java.util.PriorityQueue;

PriorityQueue<Edge> priQue = new PriorityQueue<Edge>();

In [236]:
// add() 와 offer()는 둘 다 데이터를 추가하는 기능을 제공함
priQue.add(new Edge(2, "1"));
priQue.add(new Edge(6, "2"));
priQue.offer(new Edge(5, "3"));
priQue.offer(new Edge(3, "4"));

true

In [218]:
System.out.println(priQue);


[node: 1, route: 2, node: 4, route: 3, node: 3, route: 5, node: 2, route: 6]


In [237]:
priQue.peek(); // 우선순위 큐에서 제일 위에 있는 값을 출력함

node: 1, route: 2

In [238]:
 // 우선순위 큐의 데이터를 읽음 , poll()메서드는 remove와 동일한 기능을 하며 제일 위 데이터를 읽고 삭제함
Edge edges = priQue.poll();
System.out.println(edges);
System.out.println(priQue);

node: 1, route: 2
[node: 4, route: 3, node: 2, route: 6, node: 3, route: 5]


In [239]:
 priQue.size(); // 데이터 크기 확인

3

#### 1.2.  다익스트라 알고리즘 구현 (우선순위 큐를 활용하여 구현)

- 탐색할 그래프의 시작 정점과 다른 정점들간의 최단 거리 구하기

![nn](img/img_13_02.png)

In [270]:
import java.util.HashMap;

HashMap<String, ArrayList<Edge>> s_route = new HashMap<String, ArrayList<Edge>>();
s_route.put("1", new ArrayList<Edge>(Arrays.asList(new Edge(2, "2"), new Edge(5, "3"), new Edge(1, "4"))));
s_route.put("2", new ArrayList<Edge>(Arrays.asList(new Edge(3, "3"), new Edge(2, "4"))));
s_route.put("3", new ArrayList<Edge>(Arrays.asList(new Edge(3, "2"), new Edge(5, "6"))));
s_route.put("4", new ArrayList<Edge>(Arrays.asList(new Edge(3, "3"), new Edge(1, "5"))));
s_route.put("5", new ArrayList<Edge>(Arrays.asList(new Edge(1, "3"), new Edge(2, "6"))));
s_route.put("6", new ArrayList<Edge>(Arrays.asList()));

#### Hashmap 에 들어 있는 모든 정보 가져오기

In [246]:
for(String key : s_route.keySet()) {
    System.out.println(key);
    System.out.println(s_route.get(key));
}

1
[node: 2, route: 2, node: 3, route: 5, node: 4, route: 1]
2
[node: 3, route: 3, node: 4, route: 2]
3
[node: 2, route: 3, node: 6, route: 5]
4
[node: 3, route: 3, node: 5, route: 1]
5
[node: 3, route: 1, node: 6, route: 2]
6
[]


#### 다익스트라 알고리즘 초반에 구현해야 할 것 
- 첫번째 노드를 기준으로 배열을 선언하여 처음 노드에서 각 노드까지의 가중치를 배열에 저장함
   - 시작 노드의 가중치는 0으로, 나머지 노드는 무한대로 설정함

In [247]:
// 10주차 그래프 edge 구현
public class Edge implements Comparable<Edge> {
    public int route;
    public String node;
    
    public Edge (int route, String node) {
        this.route = route;
        this.node = node;
    }
    
    public String toString() {
        return "node: " + this.node + ", route: " + this.route;
         
    }
    @Override
    public int compareTo(Edge edge) {
        return this.route - edge.route;
    }
}

In [271]:
import java.util.PriorityQueue;
import java.util.HashMap;

public class DijkstraAlgo {
    public HashMap<String, Integer> dijkstraFunction(HashMap<String, ArrayList<Edge>> s_route, String start) {
        HashMap<String, Integer> weights = new HashMap<String, Integer>();
        for (String key : s_route.keySet()) {
            weights.put(key, Integer.MAX_VALUE);
        }
        weights.put(start, 0);
    
    
    PriorityQueue<Edge> priQue = new PriorityQueue<Edge>();
    priQue.add(new Edge(weights.get(start), start));
    
    return weights;
    }

}

In [272]:
DijkstraAlgo dObject = new DijkstraAlgo();
dObject.dijkstraFunction(s_route, "1");

{1=0, 2=2147483647, 3=2147483647, 4=2147483647, 5=2147483647, 6=2147483647}

### 다익스트라 알고리즘
- 우선 순위 큐를 적용해 구현함

In [280]:
import java.util.PriorityQueue;
import java.util.HashMap;
import java.util.Arrays;
import java.util.ArrayList;

public class DijkstraAlgo {
    public HashMap<String, Integer> dijkstraFunction(HashMap<String, ArrayList<Edge>> spath, String start) {
        Edge edgeNode, adjacentNode;
        ArrayList<Edge> nodeList;
        int currentLocation, weight, route;
        String currentNode, adjacent;
        HashMap<String, Integer> weights = new HashMap<String, Integer>();
        for (String key : s_route.keySet()) {
            weights.put(key, Integer.MAX_VALUE);
        }
        weights.put(start, 0);
        
        PriorityQueue<Edge> priQue = new PriorityQueue<Edge>();
        priQue.add(new Edge(weights.get(start), start));
        
        // 알고리즘 작성
        while (priQue.size() > 0) {
            edgeNode = priQue.poll();
            currentLocation = edgeNode.route;
            currentNode = edgeNode.node;
            
            if (weights.get(currentNode) < currentLocation) {
                continue;
            }
            
            nodeList = s_route.get(currentNode);
            for (int idx = 0; idx < nodeList.size(); idx++) {
                adjacentNode = nodeList.get(idx);
                adjacent = adjacentNode.node;
                weight = adjacentNode.route;
                route = currentLocation + weight;
                
                if (route < weights.get(adjacent)) {
                    weights.put(adjacent, route);
                    priQue.add(new Edge(route, adjacent));
                }
            }
        }
        return weights;
    }
}

#### 시나리오 그래프 정의 및 실행

![nn](img/img_13_02.png)

In [281]:
HashMap<String, ArrayList<Edge>> s_route = new HashMap<String, ArrayList<Edge>>();
s_route.put("1", new ArrayList<Edge>(Arrays.asList(new Edge(2, "2"), new Edge(5, "3"), new Edge(1, "4"))));
s_route.put("2", new ArrayList<Edge>(Arrays.asList(new Edge(3, "3"), new Edge(2, "4"))));
s_route.put("3", new ArrayList<Edge>(Arrays.asList(new Edge(3, "2"), new Edge(5, "6"))));
s_route.put("4", new ArrayList<Edge>(Arrays.asList(new Edge(3, "3"), new Edge(1, "5"))));
s_route.put("5", new ArrayList<Edge>(Arrays.asList(new Edge(1, "3"), new Edge(2, "6"))));
s_route.put("6", new ArrayList<Edge>(Arrays.asList()));

In [284]:
DijkstraAlgo dObject = new DijkstraAlgo();
dObject.dijkstraFunction(s_route, "1");

{1=0, 2=2, 3=3, 4=1, 5=2, 6=4}

#### 1.3. 시간 복잡도
  
- 각 단계별 시간 복잡도
  - 각 노드는 최대 한 번씩 방문하므로, 그래프의 모든 간선은 최대 한 번씩 검사함
    - 즉, 각 노드마다 인접한 간선들을 모두 검사하는 과정은 $ O(E) $ 시간이 소요됨

  - 우선순위 큐에서 가장 많은 노드, 가중치가 들어가는 경우, 우선순위 큐에 노드/거리 정보를 넣고, 삭제하는 과정에서 최악의 시간이 발생함
    - 모든 간선이 검사될 때마다, 배열의 최단 거리가 갱신되며 추가되기 때문임
    - 이 때 최대 O(E)의 시간이 걸리고, O(E) 개의 노드/거리 정보에 대해 우선순위 큐를 유지하는 작업은 $ O(log{E}) $ 가 걸림
      - 따라서 시간 복잡도는 $ O(Elog{E}) $ 
    
따라서 총 시간 복잡도는 아래와 같음
  - $ O(E) + O(Elog{E}) = O(E + Elog{E}) = O(Elog{E}) $ 